# Intent Classification — Training Pipeline

Trains a Decision Tree on intent features derived from the target-type belief
distributions, finds the optimal temporal-fusion threshold, applies final
belief fusion, and saves the output to CSV.

All tuneable values live in **`params.py`** — edit there, not here.

## 1. Imports

In [1]:
%load_ext autoreload
%autoreload 2

from src.config import *
from src.preprocess import *
from src.model_training.models import *
from src.postprocess.intent_belief_fusion import *
from src.optimize.optimization import *

import params   # central configuration — see params.py

## 2. Load Dataset

In [2]:
data = pd.read_csv(params.INPUT_PATH, header=None)

# Intent features: columns 9000 – 12999  |  Label: last column
X = data.iloc[:, 9000:13000]
y = data.iloc[:, -1]

print(f"Dataset loaded  — X: {X.shape}, y: {y.shape}")
print(f"Unique classes  : {sorted(y.unique())}")

Dataset loaded  — X: (4000, 4000), y: (4000,)
Unique classes  : ['Attacking', 'Non Attacking']


## 3. Train / Test Split

In [3]:
train_concat, test_concat = split_dataset(
    X, y,
    test_size=params.INTENT_TEST_SIZE,
)

print(f"Train : {len(train_concat):>6,} samples")
print(f"Test  : {len(test_concat):>6,} samples")

Train :  3,200 samples
Test  :    800 samples


## 4. Sliding-Window Slicing

In [4]:
train_slices, test_slices = [
    slice_dataset(d, params.K2, params.M, params.W)
    for d in (train_concat, test_concat)
]

print(f"Train slices shape : {pd.DataFrame(train_slices).shape}")
print(f"Test  slices shape : {pd.DataFrame(test_slices).shape}")

Train slices shape : (64000, 201)
Test  slices shape : (16000, 201)


## 5. Load Target-Type Belief Distributions

Membership features for the intent branch are derived from the belief
distributions produced by `main-type.ipynb`.

In [5]:
import os
os.makedirs(params.TYPE_OUTPUT_DIR, exist_ok=True)

Train_proba = pd.read_csv(params.TYPE_TRAIN_OUTPUT_CSV, header=None)
Test_proba  = pd.read_csv(params.TYPE_TEST_OUTPUT_CSV,  header=None)

print(f"Train belief dist : {Train_proba.shape}")
print(f"Test  belief dist : {Test_proba.shape}")

Train belief dist : (64000, 12)
Test  belief dist : (16000, 12)


## 6. Feature Engineering

Membership function → rule extraction → attacker-constraint update → threat mapping.

In [6]:
# ── Membership functions from type-branch beliefs ──────────────────────────
Train_Membership = membership_function(labelling(Train_proba))
Test_Membership  = membership_function(labelling(Test_proba))

# ── Rule extraction ────────────────────────────────────────────────────────
rule_train, rule_test = [
    extract_rule(d, params.K2, params.W)
    for d in (train_slices, test_slices)
]

# ── Attacker-constraint conditioning ──────────────────────────────────────
train_update = Cond(
    pd.DataFrame(rule_train).iloc[:, :-1],
    pd.DataFrame(rule_train).iloc[:, -1],
)
test_update = Cond(
    pd.DataFrame(rule_test).iloc[:, :-1],
    pd.DataFrame(rule_test).iloc[:, -1],
)

# ── Map threat features ────────────────────────────────────────────────────
train_features = map_threat(train_update, Train_Membership)
test_features  = map_threat(test_update,  Test_Membership)

X_train = train_features.iloc[:, :-1]
y_train = train_features.iloc[:, -1]
X_test  = test_features.iloc[:, :-1]
y_test  = test_features.iloc[:, -1]

print(f"Train features : {X_train.shape}  |  Test features : {X_test.shape}")

Train features : (64000, 6)  |  Test features : (16000, 6)


## 7. Decision Tree Training

In [7]:
import os
os.makedirs(params.TRAINED_MODELS_DIR, exist_ok=True)

test_accuracy, test_prob = decision_tree(
    X_train, y_train,
    X_test,  y_test,
    save_path=params.DT_MODEL_PATH,
)

print(f"Decision Tree Test Accuracy : {test_accuracy:.6f}")

Decision Tree Accuracy : 0.92825
Decision Tree Test Accuracy : 0.928250


## 8. Optimal Temporal-Fusion Threshold

Optimises `x` (discount factor) on the **training set** to maximise
post-fusion accuracy, then stores the result back in `params.py`.

In [8]:
# ── Reload model and predict on training set ───────────────────────────────
DT_model   = joblib.load(params.DT_MODEL_PATH)
X_train_np = np.array(X_train)

train_pred     = DT_model.predict(X_train_np)
train_prob     = pd.DataFrame(DT_model.predict_proba(X_train_np))
train_accuracy = metrics.accuracy_score(y_train, train_pred)

print(f"DT Train Accuracy : {train_accuracy:.6f}")

# ── Belief mapping on training set ────────────────────────────────────────
label       = y_train.reset_index(drop=True)
belief_data = pd.DataFrame(belief_mapping_intent(train_prob, train_accuracy))

# ── Optimisation ──────────────────────────────────────────────────────────
result = minimize(
    objective,
    x0=[1.0],
    args=(belief_data, label),
    bounds=[(0, 1)],
    method='trust-constr',
)

optimal_x        = result.x[0]
optimal_accuracy = -result.fun

print(f"Optimal threshold x          : {optimal_x:.10f}")
print(f"Accuracy at optimal threshold: {optimal_accuracy:.6f}")

DT Train Accuracy : 0.951562


C:\Users\tenzi\anaconda3\envs\tf_gpu\lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


Optimal threshold x          : 0.7356230806
Accuracy at optimal threshold: 0.946359


C:\Users\tenzi\anaconda3\envs\tf_gpu\lib\site-packages\scipy\optimize\_hessian_update_strategy.py:182: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  warn('delta_grad == 0.0. Check if the approximated '


## 9. Persist Accuracies & Optimal Threshold to `params.py`

Writes `DT_ACCURACY` and `OPTIMAL_X` into `params.py` so that the
inference notebook (`demo.ipynb`) can import them without re-running training.

In [9]:
import re, pathlib

params_path = pathlib.Path("params.py")
text = params_path.read_text()

def _update(src: str, name: str, value: float) -> str:
    """Replace the `NAME: float = ...` line with the new value."""
    pattern     = rf"^({re.escape(name)}\s*:\s*float\s*=).*$"
    replacement = rf"\1 {value:.10f}"
    return re.sub(pattern, replacement, src, flags=re.MULTILINE)

text = _update(text, "DT_ACCURACY", test_accuracy)
text = _update(text, "OPTIMAL_X",   optimal_x)

params_path.write_text(text)

print("params.py updated:")
print(f"  DT_ACCURACY = {test_accuracy:.10f}")
print(f"  OPTIMAL_X   = {optimal_x:.10f}")

params.py updated:
  DT_ACCURACY = 0.9282500000
  OPTIMAL_X   = 0.7356230806


## 10. Final Belief Mapping & Temporal Fusion

In [10]:
# ── Belief mapping on test probabilities (exclude true-label column) ───────
BM_intent = belief_mapping_intent(
    test_prob.iloc[:, :-1].values,
    test_accuracy,
)

# ── Temporal fusion across time-steps ─────────────────────────────────────
combined_mass_functions = temporal_fusion_intent(
    pd.DataFrame(BM_intent),
    optimal_x,
    batch_size=params.INTENT_BATCH_SIZE,
)

True_label   = test_prob.iloc[:, -1].reset_index(drop=True)
intent_belief = pd.concat([combined_mass_functions, True_label], axis=1)

print(f"Final intent belief shape: {intent_belief.shape}")
intent_belief.head()

Final intent belief shape: (16000, 5)


,0,1,2,3,Actual
0,9.072745e-04,0.890536,0.036807,0.071750,NonHostile
1,1.404615e-04,0.987225,0.007115,0.005519,NonHostile
2,1.590822e-05,0.998550,0.001020,0.000414,NonHostile
3,1.593014e-06,0.999836,0.000131,0.000031,NonHostile
4,1.494350e-07,0.999982,0.000016,0.000002,NonHostile


## 11. Save Output

In [11]:
os.makedirs(params.INTENT_OUTPUT_DIR, exist_ok=True)

intent_belief.to_csv(
    params.INTENT_TEST_OUTPUT_CSV,
    index=False,
    header=None,
)

print(f"Saved → {params.INTENT_TEST_OUTPUT_CSV}")

Saved → C:\Users\tenzi\Desktop\MS_Research_Work\Combat Identification\output\target_intent_belief_distribution\test_belief_distribution.csv
